# MRCD Module Testing

Test từng module riêng biệt trước khi chạy full pipeline.

## Setup

In [ ]:
import sys
import os

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

## 1. Test Config & Utils

In [ ]:
from src.config import *

print(f'LLM Model: {LLM_MODEL_NAME}')
print(f'SLM Backend: {SLM_BACKEND}')
print(f'Knowledge Mode: {KNOWLEDGE_MODE}')
print(f'Data Dir: {DATA_DIR}')
print(f'Confidence Threshold: {CONFIDENCE_THRESHOLD}')
print(f'Num Rounds: {NUM_LOOP}')

In [ ]:
from src.utils import preprocess_text, clean_query, truncate_text

sample = 'BREAKING: @CNN reports http://example.com shocking news!!!'
print(f'Original: {sample}')
print(f'Preprocessed: {preprocess_text(sample)}')
print(f'Clean query: {clean_query(sample)}')
print(f'Truncated: {truncate_text(sample, 30)}')

## 2. Test Labels

In [ ]:
from src.labels import (
    generate_demo_label,
    to_clean_demo_label,
    parse_llm_label,
    REAL_SYNONYM_LABELS,
    FAKE_SYNONYM_LABELS,
)

print(f'Demo label (random): {generate_demo_label()}')
print(f'Clean demo label (real): {to_clean_demo_label(0)}')
print(f'Clean demo label (fake): {to_clean_demo_label(1)}')

# Test LLM output parsing
test_cases = ['Real', 'Fake', 'real\n', '  fake  ', 'I think Real', 'garbage']
for tc in test_cases:
    result, label = parse_llm_label(tc, return_matched_label=True)
    print(f'  "{tc}" -> {result} ({label})')

## 3. Test LLM (requires GPU)

In [ ]:
from src.llm import get_llm

llm = get_llm()
resp = llm.generate_text('What is 2+2?', max_output_tokens=16)
print(f'LLM response: {resp}')

## 4. Test Retrieval - Demo Branch

In [ ]:
from src.retrieval.demo_retrieval import search_news, load_news_corpus, retrieve_demonstrations

# Test Bing search
results = search_news('climate change science', max_results=3)
print(f'Bing results: {len(results)}')
for r in results[:2]:
    print(f'  {r[:100]}...')

## 5. Test Knowledge Retrieval (2 modes)

In [ ]:
from src.retrieval.knowledge_agent import build_knowledge_bundle

claim = 'Israel strikes Tehran and Beirut as Middle East conflict escalates.'

# Mode 1: Wiki only
kb_wiki = build_knowledge_bundle(claim, mode='wiki_only')
print('=== WIKI ONLY ===')
print(kb_wiki['combined_text'][:500])

print()

# Mode 2: Full (wiki + fact)
kb_full = build_knowledge_bundle(claim, mode='full')
print('=== FULL ===')
print(kb_full['combined_text'][:500])

## 6. Test SLM

In [ ]:
from src.slm import IntegratedSLM

slm = IntegratedSLM()
pred, conf, probs = slm.inference('This is a verified news report from Reuters.')
print(f'Prediction: {pred}, Confidence: {conf:.4f}')
print(f'Probs: {probs}')

## 7. Test Data Loading

In [ ]:
from src.slm.dataset import load_data_from_csv

train_texts, train_labels, test_texts, test_labels = load_data_from_csv()
print(f'Train: {len(train_texts)}, Test: {len(test_texts)}')
if test_texts:
    print(f'Sample: {test_texts[0][:100]}...')